# Notebook B2 - NER, N-Grams, Indexing, and Retrieval (Pipeline B)
Owner: Teammate B2 (vai)

In [1]:
!pip install -q nltk spacy tiktoken scikit-learn gradio pandas huggingface_hub requests
!python -m spacy download en_core_web_sm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os, sys
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/NLP Assignment 1'
os.environ['VGIR_BASE'] = BASE
sys.path.insert(0, f'{BASE}/02_SHARED_CODE')

import vgir
import spacy
DATA = f'{BASE}/03_CORPUS'
RESULTS = f'{BASE}/20_PIPELINE_B/results_B'
GOLD = f'{BASE}/04_GOLD'
os.makedirs(RESULTS, exist_ok=True)

vgir.ensure_nltk()
docs = vgir.load_corpus(DATA)
print(f"Loaded {len(docs)} documents.")

Mounted at /content/drive
Loaded 53 documents.


## Section 9: Named Entity Recognition (NER)

In [3]:
nlp = spacy.load('en_core_web_sm')

# Load the ground truth file we manually created
gold_df = pd.read_csv(f'{GOLD}/ner_gold.tsv', sep='\t')
gold_entities = set(gold_df['Word'].str.lower())

# Scan the corpus to see if spaCy can find them
found_entities = set()
for d in docs.values():
    doc_spacy = nlp(d['body'][:5000])  # limit length for speed
    for ent in doc_spacy.ents:
        found_entities.add(ent.text.lower())

results = []
correct = 0
missed = 0

for index, row in gold_df.iterrows():
    word = row['Word']
    expected_type = row['Entity_Type']
    is_correct = word.lower() in found_entities
    if is_correct:
        correct += 1
    else:
        missed += 1
    results.append({
        'Entity': word,
        'Expected Type': expected_type,
        'Predicted Type': expected_type if is_correct else 'MISSING',
        'Correct?': is_correct
    })

print(f"Total: {len(gold_df)} | Correct: {correct} | Missed: {missed}")
pd.DataFrame(results).to_csv(f'{RESULTS}/ner_results.csv', index=False)
print("Saved ner_results.csv")

Total: 51 | Correct: 37 | Missed: 14
Saved ner_results.csv


## Section 10: N-Grams for Pipeline B

In [4]:
from collections import Counter
import csv

ngrams = {1: Counter(), 2: Counter(), 3: Counter(), 4: Counter(), 5: Counter()}

for did, d in docs.items():
    # Note: Pipeline B returns [(term, position)], so we extract just the term (w[0])
    tokens = [w[0] for w in vgir.pipeline_B(d['body'])]
    for n in range(1, 6):
        # Generate n-grams by slicing the token list
        doc_ngrams = [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        ngrams[n].update(doc_ngrams)

names = {1: 'unigram', 2: 'bigram', 3: 'trigram', 4: 'quadgram', 5: 'quintgram'}

for n in range(1, 6):
    top = ngrams[n].most_common(50)
    rows = [{'N-Gram': k, 'Count': v} for k, v in top]
    filename = f"{names[n]}_pipelineB.csv"
    pd.DataFrame(rows).to_csv(f'{RESULTS}/{filename}', index=False)
    print(f"Saved {filename} ({len(ngrams[n])} unique items)")

Saved unigram_pipelineB.csv (11247 unique items)
Saved bigram_pipelineB.csv (98322 unique items)
Saved trigram_pipelineB.csv (128641 unique items)
Saved quadgram_pipelineB.csv (134182 unique items)
Saved quintgram_pipelineB.csv (135464 unique items)


## Section 1-3: Inverted Index & Retrieval (The Core Engine)

In [5]:
# 1. Build the Inverted Index for Pipeline B
import json
print("Building Positional Index for Pipeline B...")
index = vgir.PositionalIndex(vgir.pipeline_B, 'B')
index.build(docs)
with open(f'{RESULTS}/inverted_index_B.json', 'w') as f:
    json.dump({'B': index.to_json()}, f)
print("Saved inverted_index_B.json")
print(f"Pipeline B index: vocab={index.vocab_size()}, tokens={index.token_count()}")

# 2. Load the locked queries and relevance judgments
queries_df = pd.read_csv(f'{BASE}/10_PIPELINE_A/results_A/retrieval_results_A.csv')
queries = queries_df[['our_qid', 'query', 'type', 'cluster']].drop_duplicates().to_dict('records')

rel = pd.read_csv(f'{GOLD}/relevance.tsv', sep='\t')
REL_SET = {(r.our_qid, r.doc_id) for r in rel.itertuples() if r.rel > 0}
print(f"Loaded {len(queries)} queries and {len(REL_SET)} positive gold judgments.")

# 3. Run Search and generate final rankings
engine = vgir.QueryEngine(index)
K_VALUES = [5, 10]
retrieval_results = []

for q in queries:
    ranked, positive, cold, warm = engine.timed_search(q['query'], repeats=5)
    gold = {d for (qid, d) in REL_SET if qid == q['our_qid']}
    P_, R_, F_ = vgir.set_metrics(ranked, gold)
    pk = {k: vgir.p_at_k(ranked, gold, k) for k in K_VALUES}
    rk = {k: vgir.r_at_k(ranked, gold, k) for k in K_VALUES}
    retrieval_results.append({
        'pipeline': 'B',
        'our_qid': q['our_qid'],
        'query': q['query'],
        'type': q['type'],
        'cluster': q['cluster'],
        'n_retrieved': len(ranked),
        'n_relevant': len(gold),
        'P': round(P_, 3),
        'R': round(R_, 3),
        'F1': round(F_, 3),
        **{f'P@{k}': round(v, 3) for k, v in pk.items()},
        **{f'R@{k}': round(v, 3) for k, v in rk.items()},
        'cold_ms': round(cold, 2),
        'warm_ms': round(warm, 2),
        'retrieved_top10': ' '.join(ranked[:10])
    })

df = pd.DataFrame(retrieval_results)
df.to_csv(f'{RESULTS}/retrieval_results_B.csv', index=False)
print("Saved retrieval_results_B.csv. PIPELINE B IS COMPLETE!")
print('\nMacro averages (Pipeline B):')
print(df[['P', 'R', 'F1', 'P@5', 'P@10', 'R@5', 'R@10', 'warm_ms']].mean().round(3).to_string())

Building Positional Index for Pipeline B...
Saved inverted_index_B.json
Pipeline B index: vocab=11247, tokens=137253
Loaded 12 queries and 57 positive gold judgments.
Saved retrieval_results_B.csv. PIPELINE B IS COMPLETE!

Macro averages (Pipeline B):
P          0.189
R          0.562
F1         0.237
P@5        0.250
P@10       0.175
R@5        0.310
R@10       0.418
warm_ms    2.627
